# Assessment Notebook

## Problem 1: Representing SHA-256 Data

Before implementing SHA-256, investigate how the input, output, and intermediate data can be represented in Python.

Explain how you would represent:

    32-bit words,
    sequences of 32-bit words,
    input messages,
    512-bit message blocks,
    the final 256-bit hash value,
    big-endian integers.

For each representation, explain why it is appropriate and demonstrate your choices with small Python examples.

**32-bit words:** I represent each word as a Python `int`. Since Python integers can be larger than 32 bits, I apply a 32-bit mask (`0xFFFFFFFF`) to keep only the lowest 32 bits. This truncates any bits above the lowest 32; it does not check whether the original value was already in range.

In [6]:
MASK32 = 0xFFFFFFFF
word = 0x123456789  # This value has more than 32 bits.

# Masking truncates the value to its lowest 32 bits.
word = word & MASK32

print(f"0x{word:08x}")  # 0x23456789

0x23456789


**Sequences of 32-bit words:** I represent a sequence as a Python list of integers. The list keeps the words in order and lets me access or process them together instead of creating a separate variable for each word. Each value must still be within the unsigned 32-bit range, because a list does not enforce a bit width.

In [11]:
# Each item in the list is one 32-bit word.
words = [ 0x01234567, 0x89ABCDEF, 0x00000001,]

# Format each word with 8 hex digits, including leading zeroes.
print([f"{word:08x}" for word in words])

['01234567', '89abcdef', '00000001']


**Input messages:** I represent a text message as `bytes` after encoding it as UTF-8. Python's `input()` returns a string (`str`), while SHA-256 processes the message as bytes. Encoding the string makes the data suitable for later hashing.

In [8]:
# input() returns the message as text (a Python str).
message = input("Enter a message: ")

# Encode the text as UTF-8 bytes, the form SHA-256 processes.
message_bytes = message.encode("utf-8")

# Show the byte representation of the message.
print(message_bytes)

b'This is a message'


**512-bit message blocks:** A SHA-256 message block contains 512 bits, or 64 bytes. A Python `bytes` object is suitable for representing the block because it stores an ordered sequence of byte values. This example creates a 64-byte block; it is only demonstrating the size and representation, not the SHA-256 padding process.

In [28]:
# Begin with the bytes for "Hello" and fill the rest with zero bytes.
block = b"Hello" + bytes(59)

# A SHA-256 block is 64 bytes (512 bits).
print(len(block), "bytes")

print(len(block) * 8, "bits")

64 bytes
512 bits


**The final 256-bit hash value:** A SHA-256 digest can be represented as 32 bytes because each byte contains 8 bits, making 256 bits in total. It is often displayed as 64 hexadecimal characters, with two characters for each byte. The example below uses placeholder bytes to demonstrate the size and display format; it does not calculate a real hash.

In [27]:
# Use 32 varied placeholder bytes to represent a 256-bit digest.
# These bytes demonstrate the size and format, not a real hash result.
digest = bytes(range(32))

# A SHA-256 digest is 32 bytes (256 bits).
print(len(digest), "bytes")

# Hexadecimal displays each byte using two characters.
print(digest.hex())

32 bytes
000102030405060708090a0b0c0d0e0f101112131415161718191a1b1c1d1e1f


**Big-endian integers:** Big-endian means the most significant byte comes first. The bytes are ordered by their place value, from most significant to least significant; their numerical values do not have to decrease. Python's `int.from_bytes(data, byteorder="big")` interprets the bytes in that order and converts them to an integer.

In [30]:
# These bytes are ordered from most significant to least significant.
data = bytes([0x02, 0x04, 0x01, 0x03])

# Interpret the byte sequence as one big-endian integer.
number = int.from_bytes(data, byteorder="big")

print(hex(number))  # 0x02040103

0x2040103


## Problem 2: SHA-256 Bitwise Operations

Implement the Boolean and bitwise functions used by SHA-256:

    Ch(x, y, z)
    Maj(x, y, z)
    Sigma0(x) — written as Σ 0 256 ( x ) in the standard
    Sigma1(x) — written as Σ 1 256 ( x ) in the standard
    sigma0(x) — written as σ 0 256 ( x ) in the standard
    sigma1(x) — written as σ 1 256 ( x ) in the standard

Use NumPy so that the values involved in these operations are represented as 32-bit unsigned integers.

For each function:

    provide an appropriate docstring,
    explain the operation in Markdown,
    implement the function directly from that definition,
    provide tests on appropriate inputs.


### The SHA-256 `Ch` function

`Ch(x, y, z)` chooses bits from `y` or `z` according to the corresponding bits of `x`:

At each bit position, if `x` has a 1, the result takes that bit from `y`; if `x` has a 0, it takes it from `z`. The code converts the inputs to NumPy `uint32` values so the bitwise operations use 32-bit words.

In the example, the inputs are written with 8 bits, then padded with leading zeros when printed as 32-bit values.

In [10]:
import numpy as np

# The SHA-256 Ch function is defined as follows:
def Ch(x, y, z):
    """Return SHA-256's choose function for three 32-bit words.

    Each result bit comes from y when the corresponding bit of x is 1,
    and from z otherwise. Inputs are converted to np.uint32.
    """
    # Convert inputs to 32-bit unsigned integers to ensure proper bitwise operations.
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    # The Ch function is defined as (x AND y) XOR (NOT x AND z).
    return (x & y) ^ (~x & z)

# Example usage of the Ch function
x = 0xFFFFFFFF
y = 0x89ABCDEF
z = 0x12345678

# Call the Ch function with the example values
result = Ch(x, y, z)

# Print the result in binary format
print(f"Ch({x:08b}, {y:08b}, {z:08b}) = {result:032b}")  # Output in binary format

Ch(11111111111111111111111111111111, 10001001101010111100110111101111, 10010001101000101011001111000) = 10001001101010111100110111101111


### The SHA-256 `Maj` function

`Maj(x, y, z)` returns the majority value at each bit position: the result bit is 1 when at least two of the corresponding bits in `x`, `y`, and `z` are 1.

The expression `(x & y) ^ (x & z) ^ (y & z)` implements this rule. Each AND (`&`) finds positions where a pair of inputs both have a 1. The XOR (`^`) combines those pairwise results, giving 1 when two or three input bits are 1 and 0 otherwise. The code converts the inputs to NumPy `uint32` values so the operations use 32-bit words.

In the example, `x` is all 1s, so the majority result at each position is determined by whether `y` or `z` has a 1 there.

In [1]:
import numpy as np

# The SHA-256 Maj function is defined as follows:
def Maj(x, y, z):
    """Return SHA-256's majority function for three 32-bit words.

    Each result bit is the value shared by at least two corresponding
    input bits. Inputs are converted to np.uint32.
    """
# Convert inputs to 32-bit unsigned integers to ensure proper bitwise operations.
    x, y, z = np.uint32(x), np.uint32(y), np.uint32(z)
    # The Maj function is defined as (x AND y) XOR (x AND z) XOR (y AND z).
    return (x & y) ^ (x & z) ^ (y & z)
# Example usage of the Maj function
x = 0xFFFFFFFF
y = 0x89ABCDEF
z = 0x12345678

# Call the Maj function with the example values
result = Maj(x, y, z)

print(f"Maj({x:032b}, {y:032b}, {z:032b}) = {result:032b}")  # Output in binary format

Maj(11111111111111111111111111111111, 10001001101010111100110111101111, 00010010001101000101011001111000) = 10011011101111111101111111111111


### The SHA-256 `Sigma0` function

For a 32-bit word `x`, the SHA-256 `Σ0` function XORs three right rotations:

`Σ0(x) = ROTR²(x) XOR ROTR¹³(x) XOR ROTR²²(x)`

In the code, each rotation is built from a right shift and a left shift joined with OR (`|`). For example, shifting right by 2 and left by `32 - 2` (30) moves the two bits that fell off the right end back to the left, making a 32-bit rotation. The other shift pairs similarly implement rotations by 13 and 22 bits. Finally, XOR (`^`) combines those three rotated words. Converting `x` to `np.uint32` keeps the operation on a 32-bit word.

In [ ]:
import numpy as np

# The SHA-256 Σ0 function is defined as follows:
def Sigma0(x):
    """Return SHA-256's Σ0 function for a 32-bit word.

    The Σ0 function is defined as ROTR^2(x) XOR ROTR^13(x) XOR ROTR^22(x).
    Input is converted to np.uint32.
    """
    # Convert input to 32-bit unsigned integer to ensure proper bitwise operations.
    x = np.uint32(x)
    return np.uint32(np.right_shift(x, 2) | np.left_shift(x, 30))  ^\
            np.uint32(np.right_shift(x, 13) | np.left_shift(x, 19)) ^\
            np.uint32(np.right_shift(x, 22) | np.left_shift(x, 10))

# Example usage of the Sigma0 function
x = 0x11111111

# Call the Sigma0 function with the example value
result = Sigma0(x)

# Print the result in binary format
print(f"Sigma0({x:032b}) = {result:032b}")  # Output in binary format


Sigma0(00010001000100010001000100010001) = 10001000100010001000100010001000


## Problem 3: Generating the SHA-256 Constants

## Problem 4: Padding and Parsing Messages

## Problem 5: The SHA-256 Compression Function

## Problem 6: Complete SHA-256

# End